# Experiment 6: Input Resolution Upscaling (28×28 → 56×56)

**Single variable changed**: input image size — bicubic Resize(56) before ToTensor.
**Held constant**: architecture (DiagnosticCNN), loss (CrossEntropy), optimizer (Adam lr=0.001), epochs (15), no augmentation.

## Rationale

All five prior experiments confirm that no parametric modification (architecture, loss, augmentation, weighting) can overcome the Shirt sink at native 28×28 resolution. The root cause is resolution-limited feature extraction: ~400 foreground pixels cannot encode collar/sleeve/hemline differences among 5 upper-body classes. Upscaling to 56×56 provides 4× more pixels for the CNN to exploit — same kernel sizes operating on a finer grid.

## Architecture compatibility

DiagnosticCNN uses `AdaptiveAvgPool2d(1)` before the FC layer, making the architecture input-size agnostic. Spatial path through the network:

| Layer | 28×28 input | 56×56 input |
|-------|:-----------:|:-----------:|
| After Pool1 | 14×14 | 28×28 |
| After Pool2 | 7×7 | 14×14 |
| After Pool3 | 3×3 | 7×7 |
| After GAP | 1×1 | 1×1 |

Zero architecture changes required.

In [ ]:
import syssys.path.append('..')import os, torch, torch.nn as nn, torch.optim as optimimport numpy as npfrom src.data_utils import load_fashionmnist, get_dataloadersfrom src.train_utils import train_one_epochfrom src.eval_utils import (    evaluate_detailed, get_all_probas_and_labels,    compute_roc_auc_scores, compute_pr_auc_scores)OUT_DIR = '../outputs/error_analysis/upscale_input'os.makedirs(OUT_DIR, exist_ok=True)print(f"PyTorch: {torch.__version__}")if torch.backends.mps.is_available():    device = 'mps'elif torch.cuda.is_available():          device = 'cuda'else:                                    device = 'cpu'print(f"Device: {device}")

## Transforms — single variable change: Resize(56) added

In [ ]:
import torchvision.transforms as transforms# Upscale transform — the ONLY change from E1transform = transforms.Compose([    transforms.Resize(56, interpolation=transforms.InterpolationMode.BICUBIC),    transforms.ToTensor(),    transforms.Normalize((0.5,), (0.5,)),])train_dataset = __import__('torchvision').datasets.FashionMNIST(    root='../data', train=True, download=True, transform=transform)test_dataset = __import__('torchvision').datasets.FashionMNIST(    root='../data', train=False, download=True, transform=transform)class_names = train_dataset.classestrain_loader, test_loader = get_dataloaders(train_dataset, test_dataset, batch_size=64)print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")print(f"Train transform: {transform}")# Verify input shapesample_batch, _ = next(iter(train_loader))print(f"Input shape: {sample_batch.shape}")

## Architecture — identical to E1 DiagnosticCNN

In [ ]:
class DiagnosticCNN(nn.Module):    def __init__(self, num_classes=10):        super().__init__()        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)        self.bn1 = nn.BatchNorm2d(32)        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)        self.bn2 = nn.BatchNorm2d(32)        self.pool1 = nn.MaxPool2d(2)        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)        self.bn3 = nn.BatchNorm2d(64)        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)        self.bn4 = nn.BatchNorm2d(64)        self.pool2 = nn.MaxPool2d(2)        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)        self.bn5 = nn.BatchNorm2d(128)        self.pool3 = nn.MaxPool2d(2)        self.global_pool = nn.AdaptiveAvgPool2d(1)        self.dropout = nn.Dropout(0.3)        self.fc = nn.Linear(128, num_classes)        self.relu = nn.ReLU(inplace=True)    def get_features(self, x):        x = self.relu(self.bn1(self.conv1(x)))        x = self.relu(self.bn2(self.conv2(x)))        x = self.pool1(x)        x = self.relu(self.bn3(self.conv3(x)))        x = self.relu(self.bn4(self.conv4(x)))        x = self.pool2(x)        x = self.relu(self.bn5(self.conv5(x)))        x = self.pool3(x)        x = self.global_pool(x)        return x.view(x.size(0), -1)    def forward(self, x):        x = self.get_features(x)        x = self.dropout(x)        x = self.fc(x)        return xmodel = DiagnosticCNN().to(device)print(f"DiagnosticCNN params: {sum(p.numel() for p in model.parameters()):,}")

## Training — identical to E1 (CrossEntropyLoss, Adam, 15 epochs)

In [ ]:
criterion = nn.CrossEntropyLoss()optimizer = optim.Adam(model.parameters(), lr=0.001)num_epochs = 15train_losses = []model.train()for epoch in range(num_epochs):    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)    train_losses.append(loss)    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss:.4f}')with open(os.path.join(OUT_DIR, 'train_losses.txt'), 'w') as f:    for loss in train_losses:        f.write(f'{loss}\n')print(f"Losses saved.")

## Evaluation — identical pipeline

In [ ]:
accuracy, cm, per_class = evaluate_detailed(    model, test_loader, device, class_names, model_name='UpscaleCNN')probas, labels = get_all_probas_and_labels(model, test_loader, device, 10)roc_scores = compute_roc_auc_scores(probas, labels, model_name='UpscaleCNN')pr_scores = compute_pr_auc_scores(probas, labels, model_name='UpscaleCNN')with open(os.path.join(OUT_DIR, 'metrics_summary.txt'), 'w') as f:    f.write(f'Test Accuracy (percentage): {accuracy:.2f}\n')    f.write(f'Test Accuracy (fraction): {accuracy / 100:.4f}\n\n')    f.write(f'Macro ROC-AUC: {roc_scores["macro"]:.6f}\n')    f.write(f'Macro PR-AUC:  {pr_scores["macro"]:.6f}\n\n')    f.write(f'{"Class":<15} {"ROC-AUC":>10} {"PR-AUC":>10} {"TPR":>10} {"Precision":>10}\n')    f.write('-' * 55 + '\n')    for i, name in enumerate(class_names):        tpr = per_class[name]['TPR']        prec = per_class[name]['Precision']        f.write(f'{name:<15} {roc_scores[f"class_{i}"]:>10.4f} {pr_scores[f"class_{i}"]:>10.4f} {tpr:>10.4f} {prec:>10.4f}\n')cm_np = cm.cpu().numpy()with open(os.path.join(OUT_DIR, 'confusion_matrix.txt'), 'w') as f:    f.write(f'{"":>15}')    for name in class_names:        f.write(f'{name:>15}')    f.write('\n')    for i in range(len(class_names)):        f.write(f'{class_names[i]:>15}')        for j in range(len(class_names)):            f.write(f'{cm_np[i, j]:>15}')        f.write('\n')with open(os.path.join(OUT_DIR, 'misclassification_analysis.txt'), 'w') as f:    f.write('Misclassification Analysis\n')    f.write('=' * 70 + '\n\n')    for c in range(len(class_names)):        true_name = class_names[c]        total_errors = cm_np[c].sum() - cm_np[c, c]        f.write(f'True: {true_name}  (errors: {total_errors})\n')        f.write('-' * 50 + '\n')        for p in np.argsort(-cm_np[c]):            if p == c or cm_np[c, p] == 0:                continue            f.write(f'  -> {class_names[p]:<15} count={cm_np[c, p]:>4}\n')        f.write('\n')print(f"\nAll results saved to {OUT_DIR}/")

## Delta vs E1 Baseline

In [ ]:
def load_e1_metrics(path):    data = {}    with open(path) as f:        for line in f:            parts = line.strip().split()            if len(parts) == 5 and parts[0] != 'Class' and '-' not in line[:5]:                cls, roc, pr, tpr, prec = parts[0], float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])                data[cls] = {'tpr': tpr, 'precision': prec, 'pr_auc': pr}    return datae1 = load_e1_metrics('../outputs/error_analysis/metrics_summary.txt')print(f'{"Class":<15} {"E1 TPR":>8} {"E6 TPR":>8} {"Δ TPR":>8} {"E1 Prec":>8} {"E6 Prec":>8} {"Δ Prec":>8}')print('-' * 63)for name in class_names:    if name in e1:        e1_tpr = e1[name]['tpr']        e1_prec = e1[name]['precision']        e6_tpr = per_class[name]['TPR']        e6_prec = per_class[name]['Precision']        print(f'{name:<15} {e1_tpr:>8.3f} {e6_tpr:>8.3f} {e6_tpr - e1_tpr:>+8.3f} {e1_prec:>8.3f} {e6_prec:>8.3f} {e6_prec - e1_prec:>+8.3f}')print(f'\nAccuracy:  E1=92.50%  E6={accuracy:.2f}%  Δ={accuracy - 92.50:+.2f}%')print(f'Macro PR:   E1=0.9712  E6={pr_scores["macro"]:.4f}  Δ={pr_scores["macro"] - 0.9712:+.4f}')e6_shirt_err = cm_np[6].sum() - cm_np[6, 6]e6_tshirt_err = cm_np[0].sum() - cm_np[0, 0]e6_upper = e6_shirt_err + e6_tshirt_err + (cm_np[2].sum()-cm_np[2,2]) + (cm_np[4].sum()-cm_np[4,4]) + (cm_np[3].sum()-cm_np[3,3])print(f'\nUpper-body total errors: E6={e6_upper}')print(f'Shirt errors:   E1=153, E6={e6_shirt_err}')print(f'T-shirt errors: E1=163, E6={e6_tshirt_err}')